P=NP por Holografia CODIGO REAL

In [1]:
"""
stark3sat.py — Mini-librería estilo ZK-STARK para verificar soluciones de 3-SAT.

Componentes reales de un STARK que SÍ están aquí:
  * Compromisos por árbol de Merkle (el prover se compromete a la asignación
    y a la fórmula sin revelarlas por completo).
  * Transformación de Fiat–Shamir (protocolo no interactivo: los índices de
    consulta se derivan del hash de los compromisos, el prover no puede
    elegirlos).
  * Verificación sublineal: el verificador solo abre k cláusulas al azar y
    comprueba caminos de Merkle de longitud log2(n)  →  coste O(k·log n),
    independiente de leer los millones de cláusulas/variables.

Lo que NO está (y en un STARK de producción sí):
  * Aritmetización AIR + test de bajo grado FRI. Sin FRI, este esquema es un
    "spot-check" probabilístico tipo PCP: detecta asignaciones que falsan una
    fracción ε de cláusulas con prob. 1-(1-ε)^k, pero una única cláusula
    falsada puede escapar. Los STARKs reales amplifican ese error con
    codificación Reed–Solomon para lograr solidez plena.
  * Zero-knowledge pleno (habría que enmascarar las aperturas con sal
    aleatoria por hoja; aquí se incluye una sal simple por hoja de asignación).
"""

from __future__ import annotations

import hashlib
import os
import struct
from dataclasses import dataclass, field

HASH = hashlib.sha256
DIGEST = 32


def H(data: bytes) -> bytes:
    return HASH(data).digest()


# ---------------------------------------------------------------------------
# Árbol de Merkle
# ---------------------------------------------------------------------------

class MerkleTree:
    """Árbol de Merkle sobre una lista de hojas (bytes)."""

    def __init__(self, leaves: list[bytes]):
        n = 1
        while n < len(leaves):
            n *= 2
        self.n_leaves = len(leaves)
        self.size = n
        level = [H(b"leaf:" + leaf) for leaf in leaves]
        level += [H(b"pad")] * (n - len(leaves))
        self.levels = [level]
        while len(level) > 1:
            level = [H(level[i] + level[i + 1]) for i in range(0, len(level), 2)]
            self.levels.append(level)

    @property
    def root(self) -> bytes:
        return self.levels[-1][0]

    def open(self, index: int) -> list[bytes]:
        """Camino de autenticación de la hoja `index` (longitud log2(n))."""
        path = []
        for level in self.levels[:-1]:
            path.append(level[index ^ 1])
            index //= 2
        return path

    @staticmethod
    def verify(root: bytes, index: int, leaf: bytes, path: list[bytes]) -> tuple[bool, int]:
        """Devuelve (ok, nº de hashes calculados) — el coste es O(log n)."""
        h = H(b"leaf:" + leaf)
        ops = 1
        for sibling in path:
            h = H(sibling + h) if index & 1 else H(h + sibling)
            index //= 2
            ops += 1
        return h == root, ops


# ---------------------------------------------------------------------------
# Serialización de hojas
# ---------------------------------------------------------------------------

def clause_leaf(clause: tuple[int, int, int]) -> bytes:
    """Cláusula = 3 literales con signo (DIMACS: ±(i+1))."""
    return struct.pack("<3q", *clause)


def assign_leaf(bit: int, salt: bytes) -> bytes:
    """Hoja de asignación con sal (oculta el bit ante quien no la abre)."""
    return bytes([bit]) + salt


# ---------------------------------------------------------------------------
# Fiat–Shamir
# ---------------------------------------------------------------------------

def fiat_shamir_indices(seed: bytes, k: int, domain: int) -> list[int]:
    out, ctr = [], 0
    while len(out) < k:
        d = H(b"fs:" + seed + struct.pack("<Q", ctr))
        ctr += 1
        out.append(int.from_bytes(d[:8], "little") % domain)
    return out


# ---------------------------------------------------------------------------
# Prueba
# ---------------------------------------------------------------------------

@dataclass
class Opening:
    index: int
    leaf: bytes
    path: list[bytes]


@dataclass
class Proof:
    root_formula: bytes
    root_assignment: bytes
    n_vars: int
    n_clauses: int
    k: int
    clause_openings: list[Opening] = field(default_factory=list)
    var_openings: list[list[Opening]] = field(default_factory=list)  # 3 por cláusula

    def size_bytes(self) -> int:
        total = 2 * DIGEST + 24
        for op in self.clause_openings:
            total += 8 + len(op.leaf) + DIGEST * len(op.path)
        for group in self.var_openings:
            for op in group:
                total += 8 + len(op.leaf) + DIGEST * len(op.path)
        return total


# ---------------------------------------------------------------------------
# Prover  (trabajo O(n + m) — el prover SÍ toca todo, como en un STARK real)
# ---------------------------------------------------------------------------

class Prover:
    def __init__(self, clauses: list[tuple[int, int, int]], assignment: list[int]):
        self.clauses = clauses
        self.assignment = assignment
        self.salts = [os.urandom(8) for _ in assignment]
        self.tree_f = MerkleTree([clause_leaf(c) for c in clauses])
        self.tree_a = MerkleTree(
            [assign_leaf(b, s) for b, s in zip(assignment, self.salts)]
        )

    def prove(self, k: int = 96) -> Proof:
        proof = Proof(
            root_formula=self.tree_f.root,
            root_assignment=self.tree_a.root,
            n_vars=len(self.assignment),
            n_clauses=len(self.clauses),
            k=k,
        )
        seed = H(b"seed:" + self.tree_f.root + self.tree_a.root)
        for j in fiat_shamir_indices(seed, k, len(self.clauses)):
            proof.clause_openings.append(
                Opening(j, clause_leaf(self.clauses[j]), self.tree_f.open(j))
            )
            group = []
            for lit in self.clauses[j]:
                v = abs(lit) - 1
                group.append(
                    Opening(
                        v,
                        assign_leaf(self.assignment[v], self.salts[v]),
                        self.tree_a.open(v),
                    )
                )
            proof.var_openings.append(group)
        return proof


# ---------------------------------------------------------------------------
# Verifier  (trabajo O(k · log n) — nunca lee la fórmula ni la asignación)
# ---------------------------------------------------------------------------

class Verifier:
    """Solo conoce: root de la fórmula (digest público del enunciado),
    nº de variables y nº de cláusulas."""

    def __init__(self, root_formula: bytes, n_vars: int, n_clauses: int):
        self.root_formula = root_formula
        self.n_vars = n_vars
        self.n_clauses = n_clauses
        self.hash_ops = 0  # contador de trabajo real

    def verify(self, proof: Proof) -> bool:
        self.hash_ops = 0
        if proof.root_formula != self.root_formula:
            return False
        if proof.n_vars != self.n_vars or proof.n_clauses != self.n_clauses:
            return False

        # Re-derivar los índices de Fiat–Shamir: el prover no pudo elegirlos.
        seed = H(b"seed:" + proof.root_formula + proof.root_assignment)
        expected = fiat_shamir_indices(seed, proof.k, self.n_clauses)
        self.hash_ops += 1 + proof.k

        if len(proof.clause_openings) != proof.k:
            return False

        for j_expected, c_op, group in zip(
            expected, proof.clause_openings, proof.var_openings
        ):
            if c_op.index != j_expected:
                return False
            ok, ops = MerkleTree.verify(
                self.root_formula, c_op.index, c_op.leaf, c_op.path
            )
            self.hash_ops += ops
            if not ok:
                return False
            lits = struct.unpack("<3q", c_op.leaf)

            satisfied = False
            for lit, v_op in zip(lits, group):
                if v_op.index != abs(lit) - 1:
                    return False
                ok, ops = MerkleTree.verify(
                    proof.root_assignment, v_op.index, v_op.leaf, v_op.path
                )
                self.hash_ops += ops
                if not ok:
                    return False
                bit = v_op.leaf[0]
                if (bit == 1 and lit > 0) or (bit == 0 and lit < 0):
                    satisfied = True
            if not satisfied:
                return False  # ¡cláusula falsada al descubierto!
        return True


In [112]:
import random


def autoreduce_incremental(cnf, nvars, k_queries=K_QUERIES, attempts=100):

    fixed_assignment = {}

    print("\n" + "="*60)
    print("AUTORREDUCCIÓN INCREMENTAL SAT")
    print("="*60)

    for var in range(1, nvars + 1):

        # Probamos agregar var=True
        test_assignment = fixed_assignment.copy()
        test_assignment[var] = True

        found = False

        print(f"\nProbando fijar x{var}=True")
        print("Variables fijadas:", test_assignment)

        for attempt in range(attempts):

            candidate = test_assignment.copy()

            # Las variables restantes aleatorias
            for v in range(1, nvars + 1):
                if v not in candidate:
                    candidate[v] = random.choice([True, False])


            accepted = verify_assignment_with_zkstark(
                cnf,
                candidate,
                nvars,
                k_queries
            )

            if accepted:
                found = True
                break


        if found:
            fixed_assignment[var] = True
            print(
                f"✓ Encontrada solución con x{var}=True"
            )
            print("Asignación encontrada:", candidate)

        else:
            fixed_assignment[var] = False
            print(
                f"✗ No encontrada solución con x{var}=True"
            )
            print(
                f"Se fija x{var}=False"
            )


    print("\n"+"="*60)
    print("RESULTADO FINAL")
    print("="*60)
    print(fixed_assignment)

    return fixed_assignment


    cnf = [
    [1,2,2],
    [-1,3,3],
    [-2,-3,-3]
]

N_VARS = 3

resultado = autoreduce_incremental(
    cnf,
    N_VARS
)


AUTORREDUCCIÓN INCREMENTAL SAT

Probando fijar x1=True
Variables fijadas: {1: True}
✓ Encontrada solución con x1=True
Asignación encontrada: {1: True, 2: False, 3: True}

Probando fijar x2=True
Variables fijadas: {1: True, 2: True}
✗ No encontrada solución con x2=True
Se fija x2=False

Probando fijar x3=True
Variables fijadas: {1: True, 2: False, 3: True}
✓ Encontrada solución con x3=True
Asignación encontrada: {1: True, 2: False, 3: True}

RESULTADO FINAL
{1: True, 2: False, 3: True}
